# ReMorph OpenEnv Training Demo

This notebook is the judge-rerunnable path for the ReMorph OpenEnv submission. It validates the environment, trains/evaluates the deterministic supervised policy, checks the TRL GRPO environment-training adapter, generates plots, and leaves an optional GPU cell for a short real TRL run.

## 1. Clone and install

Use the base requirements first so CPU validation stays cheap. The optional TRL training stack is installed later only if you run the GPU cell.

In [ ]:
!git clone https://github.com/VedantPancholi/ReMorph.git
%cd ReMorph/remorph-openenv-submission
!pip install -r requirements.txt

## 2. Validate the OpenEnv package

These commands prove the manifest, smoke scenarios, deterministic inference, split generation, and OpenEnv validation path are healthy.

In [ ]:
!python -c "import yaml; yaml.safe_load(open('openenv.yaml')); print('openenv.yaml OK')"
!python scripts/smoke_test_openenv.py
!python scripts/inference.py
!python scripts/generate_splits.py --seed 42
!openenv validate

## 3. Train the supervised structured policy

This is the reliable benchmark backbone. It trains on the fixed train manifest and evaluates on the held-out eval manifest.

In [ ]:
!python scripts/train_submission.py \
  --train-manifest artifacts/submission/splits/train_manifest.json \
  --eval-manifest artifacts/submission/splits/eval_manifest.json \
  --output-dir artifacts/submission/training_run

## 4. Compare baseline, supervised, and adaptive reference

The key story: the naive baseline fails, the learned supervised policy succeeds, and the oracle/adaptive reference shows the upper bound.

In [ ]:
!python scripts/evaluate_submission.py --policy baseline --split eval \
  --train-manifest artifacts/submission/splits/train_manifest.json \
  --eval-manifest artifacts/submission/splits/eval_manifest.json
!python scripts/evaluate_submission.py --policy supervised --split eval \
  --train-manifest artifacts/submission/splits/train_manifest.json \
  --eval-manifest artifacts/submission/splits/eval_manifest.json
!python scripts/evaluate_submission.py --policy adaptive_reference --split eval \
  --train-manifest artifacts/submission/splits/train_manifest.json \
  --eval-manifest artifacts/submission/splits/eval_manifest.json

## 5. Generate production GRPO train/eval data and verify TRL plumbing

This creates seeded train/eval prompt rows (`500` train, `60` eval by default), then runs a CPU-safe dry run through the TRL `environment_factory` path and plots the dry-run metrics.

In [ ]:
!python scripts/generate_grpo_dataset.py
!python scripts/train_trl_grpo.py --dry-run --train-path artifacts/submission/grpo_dataset/train_prompts.jsonl
!python scripts/plot_trl_grpo.py

## 6. Optional: short real TRL GRPO run on GPU

Run this only on a GPU runtime. Start with 50 steps on a T4 to protect credits. Increase `--max-steps` only if reward is moving in the right direction.

In [ ]:
# !pip install -r requirements-training.txt
# !python scripts/train_trl_grpo.py \
#   --train \
#   --train-path artifacts/submission/grpo_dataset/train_prompts.jsonl \
#   --eval-path artifacts/submission/grpo_dataset/eval_prompts.jsonl \
#   --model Qwen/Qwen2.5-0.5B-Instruct \
#   --max-steps 100 \
#   --num-generations 4 \
#   --gradient-accumulation-steps 4 \
#   --repeats 4
# !python scripts/plot_trl_grpo.py

## 7. Generate final submission plots and inspect report

In [ ]:
!python scripts/generate_submission_plots.py
from pathlib import Path
print(Path('artifacts/submission/benchmark_report.md').read_text(encoding='utf-8'))

## 8. Inspect generated plot files

In [ ]:
!ls -lah artifacts/submission/plots
!ls -lah artifacts/submission/trl_grpo_run || true